# Day 8: Random Forest and XGBoost Regression

**Project:** IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning

## Objective
Train tree-based regression models for the three next-match targets and compare them against Day 7 baselines. Use chronological train/validation/test splits and keep the test set out of model selection.

## Targets
- Batting: `next_match_runs`
- Bowling: `next_match_wickets`
- Bowling: `next_match_runs_conceded`

## Important
- Use only numeric `prior_...` features reviewed on Day 7.
- Do not include current-match outcomes, targets, IDs, or future-match metadata as predictors.
- Model selection uses validation MAE/RMSE; test metrics are reported for the final comparison only.
- No results are claimed until this notebook runs successfully against your actual project data.


In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
    XGBOOST_IMPORT_ERROR = ""
except Exception as exc:
    XGBOOST_AVAILABLE = False
    XGBOOST_IMPORT_ERROR = repr(exc)

warnings.filterwarnings("ignore", category=FutureWarning)
current_dir = Path.cwd().resolve()
candidates = [current_dir, *current_dir.parents]
def is_project_root(p):
    return (p / "data" / "processed" / "batting_next_match_features.csv").is_file() or (
        (p / "data" / "processed").is_dir() and (p / "notebooks").is_dir()
    )
PROJECT_ROOT = next((p for p in candidates if is_project_root(p)), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(r"D:\Git\Personal\ipl-player-performance")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports" / "day8"
MODEL_DIR = PROJECT_ROOT / "models" / "day8"
DOCS_DIR = PROJECT_ROOT / "docs"
for folder in [REPORT_DIR, MODEL_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
BATTING_PATH = PROCESSED_DIR / "batting_next_match_features.csv"
BOWLING_PATH = PROCESSED_DIR / "bowling_next_match_features.csv"
print("Project root:", PROJECT_ROOT)
print("Batting file:", "FOUND" if BATTING_PATH.is_file() else "MISSING")
print("Bowling file:", "FOUND" if BOWLING_PATH.is_file() else "MISSING")
print("XGBoost available:", XGBOOST_AVAILABLE)
if not XGBOOST_AVAILABLE:
    print("XGBoost import issue:", XGBOOST_IMPORT_ERROR)


Project root: D:\Git\Personal\ipl-player-performance
Batting file: FOUND
Bowling file: FOUND
XGBoost available: True


In [2]:
def read_dataset(path, required):
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Complete Day 4 and review Day 7 outputs first.")
    df = pd.read_csv(path)
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} missing columns {sorted(missing)}; available: {list(df.columns)}")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["next_match_date"] = pd.to_datetime(df["next_match_date"], errors="coerce")
    return df

batting = read_dataset(BATTING_PATH, {"player_name", "match_id", "date", "next_match_date", "next_match_runs"})
bowling = read_dataset(BOWLING_PATH, {"player_name", "match_id", "date", "next_match_date",
                                     "next_match_wickets", "next_match_runs_conceded"})
print("Batting:", batting.shape)
print("Bowling:", bowling.shape)


Batting: (18030, 49)
Bowling: (14123, 50)


## 1. Feature eligibility

Recreate a conservative numeric prior-history feature list from the actual columns. The Day 7 report should still be reviewed manually; this automatic whitelist is not a substitute for verifying how the feature was constructed.


In [3]:
TARGETS = {
    "batting": ["next_match_runs"],
    "bowling": ["next_match_wickets", "next_match_runs_conceded"],
}
IDENTIFIERS = {"player_name", "player_id", "match_id", "next_match_id", "date", "next_match_date", "source_file"}
CURRENT_OUTCOMES = {"runs", "balls_faced", "fours", "sixes", "strike_rate", "batting_position", "dismissed",
                    "wickets", "runs_conceded", "legal_balls", "overs", "economy", "maidens", "dots", "wides", "noballs"}

def get_prior_features(df):
    eligible=[]
    review=[]
    for col in df.columns:
        if col in IDENTIFIERS or col in CURRENT_OUTCOMES or col in TARGETS["batting"] + TARGETS["bowling"]:
            decision="EXCLUDE"
            reason="identifier, target, future metadata, or current-match outcome"
        elif col.startswith("prior_") and pd.api.types.is_numeric_dtype(df[col]):
            decision="ELIGIBLE_CANDIDATE"
            reason="numeric prior-history feature; verify source construction"
            eligible.append(col)
        else:
            decision="REVIEW_OR_EXCLUDE"
            reason="not an approved numeric prior-history feature"
        review.append({"column":col,"dtype":str(df[col].dtype),"decision":decision,
                       "reason":reason,"missing_pct":round(df[col].isna().mean()*100,2)})
    return eligible, pd.DataFrame(review)

batting_features, batting_review = get_prior_features(batting)
bowling_features, bowling_review = get_prior_features(bowling)
feature_review_df = pd.concat([
    batting_review.assign(dataset="batting"),
    bowling_review.assign(dataset="bowling")
], ignore_index=True)
display(feature_review_df)
feature_review_df.to_csv(REPORT_DIR / "feature_eligibility_review.csv", index=False)
print("Batting candidate features:", batting_features)
print("Bowling candidate features:", bowling_features)


,column,dtype,decision,reason,missing_pct,dataset
0,match_id,int64,EXCLUDE,"identifier, target, future metadata, or curren...",0.00,batting
1,season,str,REVIEW_OR_EXCLUDE,not an approved numeric prior-history feature,0.00,batting
2,venue,str,REVIEW_OR_EXCLUDE,not an approved numeric prior-history feature,0.00,batting
3,city,str,REVIEW_OR_EXCLUDE,not an approved numeric prior-history feature,4.08,batting
4,team_1,str,REVIEW_OR_EXCLUDE,not an approved numeric prior-history feature,0.00,batting
...,...,...,...,...,...,...
94,prior_last5_mean_bowling_dots,float64,ELIGIBLE_CANDIDATE,numeric prior-history feature; verify source c...,3.56,bowling
95,prior_last5_count_bowling_dots,float64,ELIGIBLE_CANDIDATE,numeric prior-history feature; verify source c...,0.00,bowling
96,prior_career_bowling_economy,float64,ELIGIBLE_CANDIDATE,numeric prior-history feature; verify source c...,3.56,bowling
97,prior_last5_mean_bowling_economy,float64,ELIGIBLE_CANDIDATE,numeric prior-history feature; verify source c...,3.56,bowling


Batting candidate features: ['prior_career_batting_runs', 'prior_last5_mean_batting_runs', 'prior_last5_count_batting_runs', 'prior_career_batting_balls_faced', 'prior_last5_mean_batting_balls_faced', 'prior_last5_count_batting_balls_faced', 'prior_career_batting_fours', 'prior_last5_mean_batting_fours', 'prior_last5_count_batting_fours', 'prior_career_batting_sixes', 'prior_last5_mean_batting_sixes', 'prior_last5_count_batting_sixes', 'prior_career_batting_strike_rate', 'prior_last5_mean_batting_strike_rate', 'prior_last5_count_batting_strike_rate']
Bowling candidate features: ['prior_career_bowling_wickets', 'prior_last5_mean_bowling_wickets', 'prior_last5_count_bowling_wickets', 'prior_career_bowling_runs_conceded', 'prior_last5_mean_bowling_runs_conceded', 'prior_last5_count_bowling_runs_conceded', 'prior_career_bowling_legal_balls', 'prior_last5_mean_bowling_legal_balls', 'prior_last5_count_bowling_legal_balls', 'prior_career_bowling_dots', 'prior_last5_mean_bowling_dots', 'prior_

## 2. Chronological split

Use the same row-based 70% training / 15% validation / 15% test split as Day 7 for comparability. Review the date boundaries and remember that records from the same match can straddle a row boundary. If that occurs, a later improvement is to split by match date/group.


In [4]:
def split_chronologically(df, target):
    work = df.loc[df[target].notna() & df["date"].notna()].copy()
    work = work.sort_values(["date", "match_id", "player_name"], kind="stable").reset_index(drop=True)
    if len(work) < 20:
        raise ValueError(f"{target}: only {len(work)} usable rows; not enough for this split.")
    n=len(work)
    train_end=int(n*0.70)
    val_end=int(n*0.85)
    if train_end < 2 or val_end <= train_end or val_end >= n:
        raise ValueError(f"{target}: cannot form non-empty chronological splits.")
    return work, work.iloc[:train_end].copy(), work.iloc[train_end:val_end].copy(), work.iloc[val_end:].copy()

def score(y, pred):
    return {"MAE":float(mean_absolute_error(y,pred)),
            "RMSE":float(np.sqrt(mean_squared_error(y,pred))),
            "R2":float(r2_score(y,pred)) if len(y)>1 else np.nan}


## 3. Fit and compare models

The initial settings are deliberately modest and reproducible. Random Forest and XGBoost are trained on the training split; validation metrics guide selection. The test set is not used to tune parameters.


In [5]:
def evaluate_models(df, dataset_name, target, feature_cols):
    if not feature_cols:
        print(f"SKIPPED {dataset_name}/{target}: no approved prior-history features.")
        return [], None, None
    work, train, val, test = split_chronologically(df, target)
    usable=[c for c in feature_cols if train[c].notna().any()]
    if not usable:
        print(f"SKIPPED {dataset_name}/{target}: all candidate features are missing in training.")
        return [], None, None

    def X(part):
        return part[usable].apply(pd.to_numeric, errors="coerce")
    X_train, X_val, X_test = X(train), X(val), X(test)
    y_train=pd.to_numeric(train[target], errors="coerce")
    y_val=pd.to_numeric(val[target], errors="coerce")
    y_test=pd.to_numeric(test[target], errors="coerce")

    models = {
        "DummyMedian": Pipeline([("imputer",SimpleImputer(strategy="median")),
                                 ("model",DummyRegressor(strategy="median"))]),
        "LinearRegression": Pipeline([("imputer",SimpleImputer(strategy="median")),
                                      ("model",LinearRegression())]),
        "RandomForest": Pipeline([("imputer",SimpleImputer(strategy="median")),
                                  ("model",RandomForestRegressor(
                                      n_estimators=300, max_depth=None, min_samples_leaf=2,
                                      random_state=42, n_jobs=-1))])
    }
    if XGBOOST_AVAILABLE:
        models["XGBoost"] = Pipeline([("imputer",SimpleImputer(strategy="median")),
                                      ("model",XGBRegressor(
                                          n_estimators=300, max_depth=4, learning_rate=0.05,
                                          subsample=0.9, colsample_bytree=0.9,
                                          objective="reg:squarederror", random_state=42,
                                          n_jobs=-1))])

    rows=[]
    fitted={}
    for model_name, model in models.items():
        model.fit(X_train,y_train)
        fitted[model_name]=model
        for split_name, X_split, y_split in [
            ("validation",X_val,y_val),("test",X_test,y_test)
        ]:
            pred=model.predict(X_split)
            rows.append({"dataset":dataset_name,"target":target,"model":model_name,
                         "split":split_name,"feature_count":len(usable),
                         "train_rows":len(train),"validation_rows":len(val),"test_rows":len(test),
                         **score(y_split,pred)})
    split_info={
        "dataset":dataset_name,"target":target,"rows":len(work),
        "train_rows":len(train),"validation_rows":len(val),"test_rows":len(test),
        "train_start":str(train["date"].min().date()),"train_end":str(train["date"].max().date()),
        "validation_start":str(val["date"].min().date()),"validation_end":str(val["date"].max().date()),
        "test_start":str(test["date"].min().date()),"test_end":str(test["date"].max().date()),
        "features":", ".join(usable)
    }
    # Select on validation MAE only. Test scores are reported but do not determine the selection.
    validation = [r for r in rows if r["split"]=="validation"]
    best = sorted(validation, key=lambda r:(r["MAE"],r["RMSE"]))[0]["model"]
    selection={"dataset":dataset_name,"target":target,"selected_by_validation_mae":best,
               "validation_mae":next(r["MAE"] for r in validation if r["model"]==best)}
    # Persist selected fitted pipeline. It is fit on training rows only; do not treat as final production model.
    import joblib
    model_path=MODEL_DIR / f"{dataset_name}_{target}_selected_pipeline.joblib"
    joblib.dump(fitted[best],model_path)
    selection["saved_pipeline"]=str(model_path)
    return rows, split_info, selection

metric_rows=[]
split_rows=[]
selection_rows=[]
tasks=[
    (batting,"batting","next_match_runs",batting_features),
    (bowling,"bowling","next_match_wickets",bowling_features),
    (bowling,"bowling","next_match_runs_conceded",bowling_features)
]
for df, dataset_name, target, feature_cols in tasks:
    rows, split_info, selection=evaluate_models(df,dataset_name,target,feature_cols)
    metric_rows.extend(rows)
    if split_info: split_rows.append(split_info)
    if selection: selection_rows.append(selection)

metrics_df=pd.DataFrame(metric_rows)
splits_df=pd.DataFrame(split_rows)
selection_df=pd.DataFrame(selection_rows)
display(splits_df)
display(metrics_df.sort_values(["dataset","target","split","MAE"]) if not metrics_df.empty else metrics_df)
display(selection_df)
metrics_df.to_csv(REPORT_DIR / "model_comparison_metrics.csv",index=False)
splits_df.to_csv(REPORT_DIR / "chronological_split_summary.csv",index=False)
selection_df.to_csv(REPORT_DIR / "validation_model_selection.csv",index=False)


,dataset,target,rows,train_rows,validation_rows,test_rows,train_start,train_end,validation_start,validation_end,test_start,test_end,features
0,batting,next_match_runs,18030,12621,2704,2705,2008-04-18,2021-10-08,2021-10-08,2024-04-13,2024-04-13,2026-05-29,"prior_career_batting_runs, prior_last5_mean_ba..."
1,bowling,next_match_wickets,14123,9886,2118,2119,2008-04-18,2021-10-01,2021-10-01,2024-04-08,2024-04-08,2026-05-29,"prior_career_bowling_wickets, prior_last5_mean..."
2,bowling,next_match_runs_conceded,14123,9886,2118,2119,2008-04-18,2021-10-01,2021-10-01,2024-04-08,2024-04-08,2026-05-29,"prior_career_bowling_wickets, prior_last5_mean..."


,dataset,target,model,split,feature_count,train_rows,validation_rows,test_rows,MAE,RMSE,R2
3,batting,next_match_runs,LinearRegression,test,15,12621,2704,2705,16.835979,22.663426,0.101603
7,batting,next_match_runs,XGBoost,test,15,12621,2704,2705,16.918188,22.788992,0.091620
5,batting,next_match_runs,RandomForest,test,15,12621,2704,2705,17.008145,22.884593,0.083983
1,batting,next_match_runs,DummyMedian,test,15,12621,2704,2705,17.912754,26.069777,-0.188754
2,batting,next_match_runs,LinearRegression,validation,15,12621,2704,2705,15.864845,21.231884,0.105058
6,batting,next_match_runs,XGBoost,validation,15,12621,2704,2705,15.931191,21.312452,0.098254
4,batting,next_match_runs,RandomForest,validation,15,12621,2704,2705,16.176661,21.458592,0.085844
0,batting,next_match_runs,DummyMedian,validation,15,12621,2704,2705,16.475222,23.850519,-0.129310
19,bowling,next_match_runs_conceded,LinearRegression,test,15,9886,2118,2119,9.650816,12.116601,0.037079
23,bowling,next_match_runs_conceded,XGBoost,test,15,9886,2118,2119,9.767157,12.360691,-0.002108


,dataset,target,selected_by_validation_mae,validation_mae,saved_pipeline
0,batting,next_match_runs,LinearRegression,15.864845,D:\Git\Personal\ipl-player-performance\models\...
1,bowling,next_match_wickets,LinearRegression,0.789086,D:\Git\Personal\ipl-player-performance\models\...
2,bowling,next_match_runs_conceded,XGBoost,8.490233,D:\Git\Personal\ipl-player-performance\models\...


## 4. Interpretation checklist

- Compare validation MAE and RMSE for each target. Lower is generally better.
- Check test metrics only after model selection; do not pick the model based on test results.
- Compare Random Forest/XGBoost with the Day 7 baseline results.
- Inspect negative R², large errors, and unexpectedly perfect scores.
- Check feature list, split date boundaries, and the same-match boundary risk.
- The saved selected pipeline is an experiment artifact, not yet the final production model.


In [6]:
def as_text(df):
    return df.to_string(index=False) if df is not None and not df.empty else "No rows produced."

doc = f"""# Day 8: Random Forest and XGBoost Regression

## Feature review
```text
{as_text(feature_review_df)}
```

## Chronological split summary
```text
{as_text(splits_df)}
```

## Model comparison metrics
```text
{as_text(metrics_df)}
```

## Validation-based model selection
```text
{as_text(selection_df)}
```

## Outputs
- `reports/day8/feature_eligibility_review.csv`
- `reports/day8/chronological_split_summary.csv`
- `reports/day8/model_comparison_metrics.csv`
- `reports/day8/validation_model_selection.csv`
- `models/day8/*_selected_pipeline.joblib` (when a model run succeeds)
- `docs/day8_tree_based_regression.md`

## Methodology and limitations
- Target variables: `next_match_runs`, `next_match_wickets`, and `next_match_runs_conceded`.
- Only numeric `prior_...` columns are automatic feature candidates.
- Uses chronological row split: 70% train, 15% validation, 15% test.
- Median imputation is fitted inside a pipeline on training rows only.
- Selection uses validation MAE, with validation RMSE as tie-breaker; test scores do not select the model.
- The split is row-based, so a match can cross split boundaries; inspect dates and consider group-based splits before final reporting.
- XGBoost is skipped if the package cannot be imported; this does not stop Random Forest.
- The saved pipeline is trained on the training split only and is not the final production artifact.
- No metric is asserted until the notebook runs successfully against the actual project files.

## Next step
Day 9: evaluate errors, inspect feature importance, and decide a leakage-safe final training strategy.
"""
doc_path=DOCS_DIR / "day8_tree_based_regression.md"
doc_path.write_text(doc,encoding="utf-8")
print("Saved documentation:",doc_path)


Saved documentation: D:\Git\Personal\ipl-player-performance\docs\day8_tree_based_regression.md
